# Drzewa zachowań

W tym notebooku pracujemy z bibliotekami `py_trees` i `py_trees_ros`. Materiał opiera się na projekcie [`py_trees_ros_tutorials`](https://github.com/splintered-reality/py_trees_ros_tutorials), a jego kod został włączony do tej paczki warsztatowej, żeby działał w kontenerze ROS używanym na zajęciach.

Drzewo zachowań opisuje decyzje robota jako układ małych, niezależnych zachowań. Przy każdym `tick()` zachowanie zwraca jeden z czterech statusów: `SUCCESS`, `FAILURE`, `RUNNING` albo `INVALID`.

## Jak czytać wersję z rozwiązaniami

Ćwiczenia mają taką samą strukturę jak w notebooku studenckim: **Cel**, **Do zrobienia** oraz **Sprawdź**. Bezpośrednio po wybranych ćwiczeniach znajduje się sekcja `Rozwiązanie`, którą można pokazać po samodzielnej próbie albo wykorzystać jako materiał instruktorski.

In [ ]:
# Uruchom tę komórkę na początku notebooka.
# Dodaje źródła warsztatu do PYTHONPATH i w razie potrzeby przebudowuje paczkę ROS.
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROS_DISTRO = os.environ.get("ROS_DISTRO", "jazzy")
WORKSPACE = Path("/home/ubuntu/turtlebot3_ws")
SOURCE_DIR = WORKSPACE / "src" / "jupyter_notebooks"

if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))


def bash(command, check=True):
    full = f"source /opt/ros/{ROS_DISTRO}/setup.bash; " \
           f"source {WORKSPACE}/install/setup.bash 2>/dev/null || true; " \
           f"{command}"
    return subprocess.run(["bash", "-lc", full], text=True, check=check)


def ensure_ros_fun_built():
    if shutil.which("ros2") is None:
        print("Nie widzę komendy ros2. Ten notebook trzeba uruchomić w kontenerze ROS.")
        return
    marker = WORKSPACE / "install" / "ros_fun" / "lib" / "ros_fun" / "tree-data-gathering"
    if marker.exists():
        print("ros_fun ma już zainstalowane tutoriale py_trees.")
        return
    print("Buduję ros_fun, aby ros2 launch widział nowe skrypty tutoriali...")
    bash(f"cd {WORKSPACE} && colcon build --symlink-install --packages-select ros_fun")
    print("Gotowe. Terminale uruchamiane z tego notebooka będą źródłować install/setup.bash.")

ensure_ros_fun_built()


In [ ]:
import importlib.metadata

import py_trees
import py_trees_ros

print("py_trees:", importlib.metadata.version("py_trees"))
print("py_trees_ros:", importlib.metadata.version("py_trees_ros"))
print("Import py_trees_ros działa")


## Małe drzewo bez ROS

Zaczniemy od samej logiki drzewa, bez komunikacji ROS. `Selector` działa jak lista priorytetów: pierwszy sukces albo pierwsze trwające zadanie zatrzymuje sprawdzanie kolejnych gałęzi.

In [ ]:
class BatteryLow(py_trees.behaviour.Behaviour):
    def __init__(self, low=False):
        super().__init__(name="Battery low?")
        self.low = low

    def update(self):
        self.feedback_message = f"low={self.low}"
        return py_trees.common.Status.SUCCESS if self.low else py_trees.common.Status.FAILURE


def make_tree(battery_low=False):
    root = py_trees.composites.Selector(name="Robot", memory=False)
    emergency = py_trees.composites.Sequence(name="Emergency", memory=True)
    emergency.add_children([
        BatteryLow(low=battery_low),
        py_trees.behaviours.Running(name="Flash LEDs"),
    ])
    root.add_children([
        emergency,
        py_trees.behaviours.Running(name="Patrol"),
    ])
    return root

root = make_tree(battery_low=False)
print(py_trees.display.unicode_tree(root))

for i in range(3):
    root.tick_once()
    print(i, root.status, root.tip().name, root.tip().status)


---

## Ćwiczenie 1: rozbuduj gałąź awaryjną

**Cel:** zobaczyć, jak zmiana jednego warunku wpływa na wybór gałęzi w `Selector`.

**Do zrobienia:** w poprzedniej komórce zmień `battery_low=False` na `True`. Następnie dodaj do gałęzi `Emergency` nowe zachowanie po `Flash LEDs`:

```python
py_trees.behaviours.Success(name="Send warning")
```

**Sprawdź:** uruchom komórkę ponownie i upewnij się, że `Send warning` pojawia się w wydruku drzewa.

### Rozwiązanie 1

W tej wersji gałąź awaryjna zostaje uruchomiona, a po miganiu LED pojawia się dodatkowy krok `Send warning`.


In [ ]:
class BatteryLow(py_trees.behaviour.Behaviour):
    def __init__(self, low=False):
        super().__init__(name="Battery low?")
        self.low = low

    def update(self):
        self.feedback_message = f"low={self.low}"
        return py_trees.common.Status.SUCCESS if self.low else py_trees.common.Status.FAILURE


def make_tree_solution(battery_low=True):
    root = py_trees.composites.Selector(name="Robot", memory=False)
    emergency = py_trees.composites.Sequence(name="Emergency", memory=True)
    emergency.add_children([
        BatteryLow(low=battery_low),
        py_trees.behaviours.Running(name="Flash LEDs"),
        py_trees.behaviours.Success(name="Send warning"),
    ])
    root.add_children([
        emergency,
        py_trees.behaviours.Running(name="Patrol"),
    ])
    return root

root = make_tree_solution(battery_low=True)
print(py_trees.display.unicode_tree(root))
for i in range(3):
    root.tick_once()
    print(i, root.status, root.tip().name, root.tip().status)


## Drzewo ROS: dane z baterii na blackboardzie

W tutorialu 1 zachowanie `Battery2BB` subskrybuje temat `/battery/state` i zapisuje odczyt baterii na blackboardzie. Dzięki temu pozostałe zachowania korzystają ze spójnej kopii danych z bieżącego tyknięcia drzewa.

<img src="./images/py_trees_ros_tutorials/tutorial-one-data-gathering.gif" width="70%">


In [ ]:
from ros_fun_py_trees_ros_tutorials.one_data_gathering import tutorial_create_root as create_data_gathering_tree

root = create_data_gathering_tree()
print(py_trees.display.unicode_tree(root))


In [ ]:
from run_in_term import run_lxterminal


def ros(command, check=True):
    return bash(command, check=check)


def terminal(command):
    escaped = command.replace("'", "'\\''")
    run_lxterminal(
        f"bash -lc 'source /opt/ros/{ROS_DISTRO}/setup.bash; "
        f"source {WORKSPACE}/install/setup.bash; {escaped}'"
    )


def stop_py_trees_tutorials():
    pattern = "[r]os2 launch ros_fun|[p]y-trees-tree-watcher|[p]y-trees-blackboard-watcher|[t]ree-data-gathering|[t]ree-battery-check|[t]ree-action-clients|[t]ree-context-switching|[t]ree-docking-cancelling-failing|[t]ree-dynamic-application-loading|[m]ock-battery|[m]ock-dashboard|[m]ock-led-strip|[m]ock-docking-controller|[m]ock-move-base|[m]ock-rotation-controller|[m]ock-safety-sensors"
    ros(f"pkill -TERM -f '{pattern}' || true", check=False)
    ros("sleep 1", check=False)
    ros(f"pkill -KILL -f '{pattern}' || true", check=False)


In [ ]:
stop_py_trees_tutorials()
terminal("ros2 launch ros_fun tutorial_one_data_gathering_launch.py")


In [ ]:
# Po chwili od startu tutoriala sprawdź topic z baterią.
ros("ros2 topic list | grep battery", check=False)
ros("ros2 topic echo --once /battery/state", check=False)


## Priorytet: niski poziom baterii

Tutorial 2 dodaje gałąź awaryjną. Gdy `Battery2BB` zapisze na blackboardzie `battery_low_warning=True`, drzewo wybiera gałąź `Battery Low?` i publikuje komendę dla paska LED.

<img src="./images/py_trees_ros_tutorials/tutorial-two-battery-check.png" width="70%">


In [ ]:
from ros_fun_py_trees_ros_tutorials.two_battery_check import tutorial_create_root as create_battery_check_tree

root = create_battery_check_tree()
print(py_trees.display.unicode_tree(root))


In [ ]:
stop_py_trees_tutorials()
terminal("ros2 launch ros_fun tutorial_two_battery_check_launch.py")


In [ ]:
# Wymuś niski poziom baterii. Pasek LED powinien dostać komendę "red".
ros("ros2 param set /battery charging_percentage 25.0", check=False)
ros("sleep 2; timeout 5 ros2 topic echo --once /led_strip/display", check=False)


---

## Ćwiczenie 2: zmień próg i kolor ostrzeżenia

**Cel:** przećwiczyć małą zmianę w zachowaniu ROS i sprawdzić jej efekt przez temat `/led_strip/display`.

**Do zrobienia:** w pliku `ros_fun_py_trees_ros_tutorials/two_battery_check.py` znajdź `threshold=30.0` oraz `colour="red"`. Zmień próg ostrzeżenia na `50.0`, a kolor ostrzeżenia na `yellow`.

**Sprawdź:** przebuduj paczkę `ros_fun` albo uruchom pierwszą komórkę notebooka ponownie. Potem wystartuj tutorial jeszcze raz, ustaw niski poziom baterii i sprawdź, czy na `/led_strip/display` pojawia się `yellow`.

### Rozwiązanie 2

W pliku `ros_fun_py_trees_ros_tutorials/two_battery_check.py` zmień tworzenie `Battery2BB` i `FlashLedStrip` tak, aby docelowy fragment wyglądał tak:

```python
battery2bb = py_trees_ros.battery.ToBlackboard(
    name="Battery2BB",
    topic_name="/battery/state",
    qos_profile=py_trees_ros.utilities.qos_profile_unlatched(),
    threshold=50.0,
)
flash_led_strip = behaviours.FlashLedStrip(
    name="FlashLEDs",
    colour="yellow",
)
```

Po zmianie uruchom pierwszą komórkę notebooka albo `colcon build --symlink-install --packages-select ros_fun`, a następnie ustaw poziom baterii poniżej 50%.


## Sprzątanie

Po zakończeniu pracy zatrzymaj procesy uruchomione przez tutoriale.

In [ ]:
stop_py_trees_tutorials()
